<a href="https://colab.research.google.com/github/addamit/py-howtos/blob/master/ClusteringConversations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# !pip install sentence-transformers
# !pip install datasets

In [59]:
import numpy as np
import pandas as pd
import json
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


In [15]:
# Load dataset
dataset = load_dataset("determined-ai/consumer_complaints_medium", split="train")
conversations = [sample for sample in dataset.select(range(200))]  # First 200 convos


In [19]:
df = pd.json_normalize(conversations)

In [20]:

df.head()

,Issue,Consumer Complaint
0,Incorrect information on credit report,I have not been living at the address which ju...
1,"Managing, opening, or closing account",I have the Rush card and have not been able to...
2,Problems caused by my funds being low,Suntrust covered a debit card transaction when...
3,Incorrect information on credit report,I wrote Equifax on XXXX/XXXX/15 and alerted th...
4,Problem with a credit reporting company's inve...,"Over two months ago, I asked Equifax to invest..."


In [24]:
titles, docs = list(df['Issue']), list(df['Consumer Complaint'])

In [21]:
embedding_model = SentenceTransformer("all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [70]:
import torch
# Delete the model and tokenizer
del model
del tokenizer
del pipe
# Clear the cache
torch.cuda.empty_cache()

# Trigger garbage collection
gc.collect()

NameError: name 'model' is not defined

In [71]:
model_path = "microsoft/Phi-3.5-mini-instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_path)



config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

configuration_phi3.py:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_phi3.py:   0%|          | 0.00/73.8k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3.5-mini-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer_config.json:   0%|          | 0.00/3.98k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [72]:
# Create a text generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)


Device set to use cuda:0


In [81]:
# Check for context size attributes
if hasattr(model.config, 'max_position_embeddings'):
    print(f"Maximum sequence length : {model.config.max_position_embeddings}")
elif hasattr(model.config, 'n_positions'):
    print(f"Maximum sequence length: {model.config.n_positions}")
else:
    print("Context size attribute not found.")

Maximum sequence length : 131072


In [82]:

def estimate_token_count(texts):
    """Estimate total tokens in a list of texts."""
    return sum(len(tokenizer.encode(t)) for t in texts)

In [73]:
def generate_cluster_name_description(in_cluster_samples, out_cluster_samples):
    """
    Uses the HF model to generate a name and description that defines
    the in-cluster samples while differentiating them from out-cluster samples.

    Args:
        in_cluster_samples (list of str): 50 summaries from within the cluster.
        out_cluster_samples (list of str): 50 summaries from outside but near the cluster.

    Returns:
        dict: {"name": str, "description": str}
    """
    prompt = (
        f"Below are two sets of text summaries:\n\n"
        f"**In-cluster summaries:**\n{in_cluster_samples}\n\n"
        f"**Out-cluster summaries:**\n{out_cluster_samples}\n\n"
        f"Generate a short but descriptive name and a summary that captures the theme of "
        f"the in-cluster summaries while clearly distinguishing them from the out-cluster summaries."
    )

    # Define the messages for the pipeline
    messages = [
        {"role": "system", "content": "You are a helpful AI assistant."},
        {"role": "user", "content": prompt},
    ]

    # Define generation arguments
    generation_args = {
        "max_new_tokens": 250,
        "return_full_text": False,
        "temperature": 1.0,
        "do_sample": True,
    }

    # Generate the response
    output = pipe(messages, **generation_args)
    response = output[0]['generated_text']

    # Extract name and description from the response
    # Assuming the response format is consistent and can be split into name and description
    name = "Generated Name"  # Placeholder for name extraction logic
    description = response

    return {"name": name, "description": description}


In [25]:
embeddings = embedding_model.encode(docs, batch_size=32, show_progress_bar=True) #, normalize_embeddings=True)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

In [26]:

embeddings.shape

(200, 768)

In [27]:
TARGET_CLUSTER_COUNT = 5
M_NEAREST = 5
AVG_CLUSTERS_PER_NEIGHBORHOOD = 40

In [33]:
level = 0
cluster_map = {i: {"name": docs[i], "children": {}} for i in range(len(docs))}
cluster_texts = [cluster_map[i]["name"] for i in cluster_map.keys()]
cluster_embeddings = embedding_model.encode(cluster_texts, normalize_embeddings=True)

In [32]:
cluster_map[10]

{'name': 'On XXXX XXXX, Equifax, XXXX notified the public of a massive data breach, in which they have informed me that my personal data may have been compromised. As XXXX XXXX XXXX have a duty to maintain the privacy and security of the data they hold, they have failed in this duty. I have placed holds on all of my credit reports to prevent identity theft, but that may not prevent all detrimental actions.',
 'children': {}}

In [34]:
num_neighborhoods = max(1, round(len(cluster_texts) / AVG_CLUSTERS_PER_NEIGHBORHOOD))

In [35]:
num_neighborhoods

5

In [36]:
kmeans = KMeans(n_clusters=num_neighborhoods, random_state=42, n_init=10)
neighborhood_labels = kmeans.fit_predict(cluster_embeddings)

In [39]:
neighborhoods = {i: [] for i in range(num_neighborhoods)}
for idx, label in enumerate(neighborhood_labels):
    neighborhoods[label].append(idx)

In [42]:
similarity_matrix = cosine_similarity(cluster_embeddings)

In [47]:
expanded_neighborhoods = {}
for n_id, cluster_indices in neighborhoods.items():

      other_clusters = list(set(range(len(cluster_embeddings))) - set(cluster_indices))

      external_nearest = []

      for idx in cluster_indices:
          similarities = [(other_idx, similarity_matrix[idx][other_idx]) for other_idx in other_clusters]
          similarities.sort(key=lambda x: x[1], reverse=True)
          external_nearest.extend([s[0] for s in similarities[:M_NEAREST]])

      expanded_neighborhoods[n_id] = list(set(cluster_indices + external_nearest))


In [52]:
len(expanded_neighborhoods[2]), len(neighborhoods[2])

(110, 40)

In [53]:
new_cluster_map = {}
for n_id, cluster_indices in expanded_neighborhoods.items():
    in_cluster_samples = [cluster_map[list(cluster_map.keys())[idx]]["name"] for idx in cluster_indices[:50]]
    out_cluster_samples = [cluster_map[list(cluster_map.keys())[idx]]["name"] for idx in cluster_indices[-50:]]

    # cluster_info = generate_cluster_name_description(in_cluster_samples, out_cluster_samples)
    # new_cluster_name = cluster_info.split("\n")[0]
    # new_cluster_description = "\n".join(cluster_info.split("\n")[1:])

    # new_cluster_map[n_id] = {"name": new_cluster_name, "description": new_cluster_description, "children": {}}
    # for idx in cluster_indices:
    #     old_cluster_id = list(cluster_map.keys())[idx]
    #     new_cluster_map[n_id]["children"][old_cluster_id] = cluster_map[old_cluster_id]


In [65]:
test_in_cluster_samples, test_out_cluster_samples = in_cluster_samples[0], out_cluster_samples[0]
len(test_in_cluster_samples), len(test_out_cluster_samples)

(155, 218)

In [67]:
test_in_cluster_samples, test_out_cluster_samples

("I have the Rush card and have not been able to access my money for weeks. My bills have not been paid at home and my daughter college funds ca n't be paid.",
 'I am XXXX. I have been struggle to meet the payments since I have XXXX job that I have been working and still struggle to make the full payment. I have call and it take long time because I was using the relay operator.')

In [74]:
generate_cluster_name_description(test_in_cluster_samples, test_out_cluster_samples)

The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


{'name': 'Generated Name',
 'description': ' Name: Financial Strain and Access Issues\n\nSummary:\nThis theme encompasses individuals confronting critical financial strain due to unpaid bills and compromised access to funds, with a distinct divergence from out-cluster peers who primarily struggle with making payments due to lower income levels and prolonged communication delays associated with relay services. The in-cluster situation underscores the urgency of unresolved financial obligations that impact essential areas such as housing maintenance and educational expenses, whereas the out-cluster focuses on the challenges posed by systemic inefficiencies in service interactions.'}

In [75]:
test_in_cluster_samples, test_out_cluster_samples = in_cluster_samples[10], out_cluster_samples[10]
len(test_in_cluster_samples), len(test_out_cluster_samples)

(437, 359)

In [76]:
test_in_cluster_samples, test_out_cluster_samples

('i like fia and do not want to do this. FIA card services i asked for credit line increrase on card ending XXXX was declined for reasons not accurate. i have not asked before now. and i do pay. i was granted XXXX on another card recently. with it this way i will not use it much. these reasons are inaccurate. i ask for someone to check it since i can not wait on hold all day i wanted XXXX line now there is a hit on my credit card line.',
 'I sold my home last year and BBVA COMPASS BANK reported BANKRUPTCY on my credit report on XX/XX/XXXX. I have contacted COMPASS BANK numerous times and no one has returned my phone call. I have had to hire a credit company to try and resolve this issue. We sold our home in XX/XX/XXXX and purchase our new home in XX/XX/XXXX. I have never filed for Bankruptcy.')

In [77]:
generate_cluster_name_description(test_in_cluster_samples, test_out_cluster_samples)

{'name': 'Generated Name',
 'description': " **Name:** Credit Line Adjustment Concerns and Customer Service Experiences\n\n**Summary:**\nThe in-cluster summaries revolve around individuals experiencing difficulties in adjusting their existing credit lines with FIA card services, including denied requests due to alleged inaccuracies and frustrations with the response time and lack of assistance. In contrast, the out-cluster summaries detail a separate issue where an individual experiences repeated difficulties with COMPASS BANK reporting a bankruptcy on their credit report, despite the customer's claims of never filing for bankruptcy and lack of communication from the bank's representatives. These accounts share themes of credit management and customer service struggles but address distinctly different financial concerns."}

In [84]:
estimate_token_count(test_in_cluster_samples)

437